# 02. LangChain 입문 — 첫 번째 에이전트

LangChain v1의 핵심 API로 도구를 갖춘 에이전트를 만들어 봅니다.

## 학습 목표

- `@tool` 데코레이터로 커스텀 도구를 정의한다
- `create_agent()`로 에이전트를 생성한다
- `invoke()`로 에이전트를 실행하고 결과를 확인한다

## 2.1 환경 설정

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI
# 사내 LLM 모델을 호출 할때는 다른 방식 
model = ChatOpenAI(model="gpt-5.4")
print("\u2713 모델 준비 완료")

✓ 모델 준비 완료


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — 


## 2.2 도구 만들기

`@tool` 데코레이터를 붙이면 일반 함수가 에이전트 도구가 됩니다.

도구를 정의할 때 알아야 할 핵심 규칙:
- **타입 힌트(Type Hints)**: 함수 파라미터의 타입 힌트가 도구의 입력 스키마를 자동으로 정의합니다. 예를 들어 `a: int`는 정수형 입력임을 모델에 알려줍니다.
- **Docstring**: 함수의 docstring이 도구 설명(description)으로 사용됩니다. 모델은 이 설명을 보고 어떤 도구를 사용할지 결정하므로, 명확하고 간결하게 작성해야 합니다.
- **커스텀 이름/설명**: `@tool("custom_name", description="...")` 형태로 도구 이름과 설명을 직접 지정할 수도 있습니다.
- **복잡한 입력**: Pydantic `BaseModel`과 `Field`를 사용하여 복잡한 입력 스키마를 정의할 수 있습니다.

In [3]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """두 수를 더합니다."""
    print("[TOOL CALL] add")
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """두 수를 곱합니다."""
    print("[TOOL CALL] multiply")
    return a * b

print("도구 목록:")
for t in [add, multiply]:
    print(f"  - {t.name}: {t.description}")

도구 목록:
  - add: 두 수를 더합니다.
  - multiply: 두 수를 곱합니다.


## 2.3 에이전트 생성 & 실행

`create_agent()`는 모델과 도구를 결합하여 에이전트를 만듭니다.
에이전트는 내부적으로 **ReAct(Reasoning + Acting) 루프**를 실행합니다:

```
질문 → 모델이 도구 호출 결정 → 도구 실행 → 결과 관찰 → 반복 또는 최종 응답 생성
```

에이전트의 핵심 구성 요소:
- **모델(Model)**: LLM이 어떤 도구를 호출할지 판단합니다. 문자열(`"openai:gpt-5"`) 또는 모델 객체를 전달할 수 있습니다.
- **도구(Tools)**: 에이전트가 수행할 수 있는 액션입니다. 단순 바인딩과 달리 순차 호출, 병렬 실행, 재시도 등을 지원합니다.
- **시스템 프롬프트(System Prompt)**: 에이전트의 행동을 안내하는 지침입니다.

에이전트는 도구를 여러 번 순차적으로 호출하거나, 여러 도구를 병렬로 실행할 수도 있습니다. 최종 응답을 생성하거나 반복 횟수 제한에 도달하면 루프가 종료됩니다.

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[add, multiply],
    system_prompt="당신은 수학 도우미입니다.",
)
print("\u2713 에이전트 생성 완료")

✓ 에이전트 생성 완료


In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "15 + 27은 얼마인가요?"}]},
    # Langfuse 추적을 위해 config 전달 (선택)
    config=lf_config,
)
print("에이전트 응답:", result["messages"][-1].content)

에이전트 응답: 42입니다.


In [6]:
# 도구를 두 번 사용하는 복합 질문
result = agent.invoke(
    {"messages": [{"role": "user", "content": "6과 7을 곱한 다음 10을 더하세요."}]},
    config=lf_config,
)
print("에이전트 응답:", result["messages"][-1].content)

[TOOL CALL] multiply
[TOOL CALL] add
에이전트 응답: 정답은 52입니다.


## 요약

| 핵심 API | 역할 |
|---|---|
| `@tool` | 함수를 에이전트 도구로 변환 |
| `create_agent()` | 모델 + 도구 → 에이전트 생성 |
| `agent.invoke()` | 에이전트 실행, 결과 반환 |

### 다음 단계
→ **[03_langchain_memory.ipynb](./03_langchain_memory.ipynb)**: 멀티턴 대화와 메모리를 배웁니다.